# Трансферное обучение для классификации джетов в физике высоких энергий: EDA

Это первый сопроводительный ноутбук проекта по трансферному обучению. В нем представлены релевантные визуализации входных данных.

> ⚠️ **Примечание о данных**
>
> Используемый в проекте набор данных (ATLAS Top Tagging Open Data Set)
> имеет большой объем и не включен в репозиторий.
> Исходные `.h5` файлы не загружаются на GitHub и не обрабатываются
> на локальной машине из-за ограничений по ресурсам.
>
> Данный ноутбук используется как отчет по EDA:
> все визуализации и выводы были получены ранее на суперкомпьютере и сохранены
> для анализа и документирования результатов.
>
> Поэтому ноутбук предназначен для просмотра, а не для воспроизводимого запуска.
>
> Графики и результаты EDA приведены в `results/eda.ipynb`.

## Предпосылки

Эксперименты на ускорителях частиц в ЦЕРН генерируют огромные объемы данных, с обработкой которых традиционные методы анализа без использования машинного обучения зачастую не справляются. Часть этих экспериментов посвящена изучению джетов - потоков вторичных частиц, возникающих при высокоэнергетических столкновениях первичных частиц.

Для каждого джета требуется как можно быстрее определить, какая первичная частица стала причиной его образования, чтобы принять решение о сохранении соответствующего события для дальнейшего анализа.

Потоки данных, получаемые в таких экспериментах, содержат информацию высокого качества, хранение которой требует значительных вычислительных ресурсов и объемов памяти. В связи с этим возникает следующий вопрос:
возможно ли обучить student-нейросеть, принимающую на вход сниженное по качеству представление джетов, так чтобы она с той же точностью предсказывала первичную частицу, что и более сложная Teacher-модель, обученная на данных высокого качества?

В более простой формулировке, в данной работе решается задача **бинарной классификации**: требуется определить, был ли джет инициирован **топ-кварком** (`label = 1`) или относится к **фоновым** событиям (`label = 0`).

Используется набор данных [ATLAS Top Tagging Open Data Set](https://opendata.cern.ch/record/15013), представляющий собой симулированные события с джетами, подготовленные коллаборацией ATLAS.

In [1]:
import warnings

import matplotlib.pyplot as plt
import seaborn as sns
from helper import *
from preprocess import *
from visualize import *

warnings.filterwarnings("ignore")  # pesky divide-by-zero errors

sns.set_theme()
plt.style.use("seaborn-v0_8")

In [2]:
# Initialize some global hyperparameters

MAX_ITEMS = 50000  # maximum number of elements to consider (input size): 3,000,000
MAX_CONSTITS = 80  # maximum number of constituent elements per jet: 80

### Визуализация входных данных

В качестве самого первого шага загрузим данные и посмотрим, как они устроены. Наши данные имеют три атрибута: `jet`, `constituents` и `high-level`, содержащие информацию о каждом джете.

Атрибут `jet` хранит информацию обо *всем* джете целиком — например, массу и импульс джета. 

Атрибут `constituents`, в свою очередь, содержит информацию о *каждой отдельной* частице, из которых состоит джет, — например, массу и импульс каждой частицы.

Наконец, атрибут `high-level` включает переменные, «выбранные в двух отдельных исследованиях высокоуровневых величин для джет-теггеров», проведенных коллаборацией ATLAS (см. [справочное руководство](https://gitlab.cern.ch/atlas/ATLAS-top-tagging-open-data/-/tree/master?ref_type=heads)). Эти высокоуровневые признаки не являются критически важными для текущей задачи, поэтому в дальнейшем модель будет обучаться на признаках уровня конституентов, которые содержат наибольшее количество информации для идентификации джетов.

In [3]:
# jet_data, jet_labels, jet_weights, jet_features = get_data(
#     "./data/reduced_atlas_dataset.h5",
#     attribute="jet",
# )

teacher_data, teacher_labels, teacher_weights, teacher_features = get_data(
    "../data/test.h5",
    attribute="constituents",
    max_items=MAX_ITEMS,
    who="Teacher",
)

# To avoid contamination, the student data is independent of the teacher data
student_data, student_labels, student_weights, student_features = get_data(
    "../data/test.h5",
    attribute="constituents",
    max_items=MAX_ITEMS,
    who="Student",
)

# print("---------- Jet-level data ----------------")
# print("Data shape [input_size, num_features]:", jet_data.shape)
# print("Feature names:", [human_feature(f) for f in jet_features],"\n")

print("---------- Constituent-level data ----------------")
print("Data shape [input_size, num_constituents, num_features]:", teacher_data.shape)
print("Feature names:", [human_feature(f) for f in teacher_features])

---------- Constituent-level data ----------------
Data shape [input_size, num_constituents, num_features]: (50000, 200, 4)
Feature names: ['constituent transverse momentum', 'constituent pseudo-rapidity', 'constituent azimuthal angle', 'constituent energy']


Как видно, используемые признаки содержат важную информацию о джетах, такую как масса, энергия, импульс и другие характеристики. Для наглядности можно визуализировать распределения этих признаков с помощью гистограмм:

In [4]:
# plot_1D_distributions(jet_data, jet_labels, jet_features, nbins=40, transparent=False,
#                       # save_path="./jet_distribution.png",
#                       )

Данная гистограмма показывает распределения массы джета, псевдобыстроты, азимутального угла и энергии в зависимости от метки класса. Уже на этом уровне можно визуально заметить определенные различия между классами. Теперь рассмотрим, как выглядят гистограммы признаков на уровне конституентов:

In [5]:
# plot_1D_distributions(cons_data, cons_labels, cons_features, nbins=50, transparent=False,
#                       # save_path="./constituent_distribution.png",
#                       )

На уровне конституентов различать сигнал и фон становится значительно сложнее (даже визуально). Остается надеяться, что нейронная сеть сможет выявить эти тонкие различия и помочь в решении задачи классификации.

### Предобработка

Сырые данные на уровне конституентов непригодны для непосредственного обучения глубокой нейронной сети. Во-первых, необходимо учесть широкий динамический диапазон значений энергии и импульса. Во-вторых, признаки $\eta$ и $\phi$ в исходном виде содержат ограниченное количество информации и требуют преобразования с использованием физически обоснованных формул.

Данная предобработка выполняется с помощью функции `preprocess.constituent_preprocess`, напрямую адаптированной из кода, предоставленного коллаборацией ATLAS.

In [6]:
# # First, split the teacher data in half -- half HQ data, half LQ data
# teacher_hq = teacher_data[:int(MAX_ITEMS/2)]
# teacher_lq = teacher_data[int(MAX_ITEMS/2):]

# # Pass the teacher_lq array throught the same data degradation steps as the student's data will be
# teacher_lq = diffuse(teacher_lq, teacher_features, noise_std=NOISE_STD,
#                              apply_features=DIFFUSE_AXES,
#                             )

# # Preprocess LQ data
# teacher_lq, _ = constituent_preprocess(teacher_lq, teacher_features, max_constits=REDUCED_CONSTITS)
# teacher_lq = add_constits(teacher_lq, target_shape=MAX_CONSTITS)  # padding

# # Preprocess HQ data
# teacher_hq, _ = constituent_preprocess(teacher_hq, teacher_features, max_constits=REDUCED_CONSTITS)

# # Finally, stitch them pack together
# teacher_data = np.concatenate((teacher_hq, teacher_lq))

# print("Preprocessed teacher data shape:", teacher_data.shape)
# print("Preprocessed features:", teacher_features)


In [7]:
teacher_data, teacher_features = constituent_preprocess(
    teacher_data, teacher_features, max_constits=MAX_CONSTITS
)
student_data, _ = constituent_preprocess(student_data, student_features, max_constits=MAX_CONSTITS)

print("Preprocessed teacher data shape:", teacher_data.shape)
print("Preprocessed features:", teacher_features)

Preprocessed teacher data shape: (50000, 80, 7)
Preprocessed features: ['delta_eta' 'delta_phi' 'log_pt' 'log_E' 'lognorm_pt' 'lognorm_E' 'R']


Попробуем также визуализировать полученные после предобработки признаки!

In [8]:
# plot_preprocessed_1D_distributions(pre_cons_data, cons_labels, pre_cons_features, nbins=40)

На этом этапе представление данных становится менее интуитивным, однако основная идея заключается в том, что такие преобразованные признаки должны помочь нейронной сети обучаться более эффективно и достигать лучшего качества.